# Config info

In [1]:
# =============================================================================
# Load and display config info (notebook cell)
# =============================================================================

import sys
sys.path.insert(0, "..")
import utils

# --------
# Load config
# --------
config = utils._load_config()

# --------
# Extract database info
# --------
db_path        = config["database"]["db_path"]
tables_config  = config["database"]["tables"]
   
table_ohlcv       = tables_config["ohlcv"]
table_features    = tables_config["features"]
table_predictions = tables_config["predictions"]

# --------
# Display
# --------
print(f"{'='*70}")
print("📂 DATABASE CONFIGURATION")
print(f"{'='*70}")

print(f"\n🗄️  Database Path:")
print(f"   {db_path}")

print(f"\n📊 Tables:")
print(f"   [ohlcv]        {table_ohlcv}")
print(f"   [features]     {table_features}")
print(f"   [predictions]  {table_predictions}")

print(f"\n{'='*70}\n")

📂 DATABASE CONFIGURATION

🗄️  Database Path:
   D:/repos/chronoquant/data/bchusdt_data.db

📊 Tables:
   [ohlcv]        bchusdt_1m
   [features]     bchusdt_1m_features
   [predictions]  bchusdt_1m_predictions




# DB tables info

In [2]:
# =============================================================================
# Query all tables from database: row count, min/max open_time
# =============================================================================

import sys
sys.path.insert(0, "..")
import utils
import sqlite3
import pandas as pd

# --------
# Load config
# --------
config  = utils._load_config()
db_path = config["database"]["db_path"]

# --------
# Query all tables from database
# --------
print(f"\n{'='*80}")
print("📊 DATABASE TABLES SUMMARY")
print(f"{'='*80}")
print(f"{'Table Name':<25} {'Rows':<12} {'Min Date':<20} {'Max Date':<20}")
print("-" * 80)

with sqlite3.connect(db_path) as conn:
	# Get all table names from database
	tables_df = pd.read_sql_query(
		"SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
		conn
	)
	tables_list = tables_df["name"].tolist()

	# Query each table
	for table_name in tables_list:
		# --------
		# Row count
		# --------
		count_df = pd.read_sql_query(
			f"SELECT COUNT(*) as cnt FROM {table_name}",
			conn
		)
		row_count = count_df["cnt"].iloc[0]

		# --------
		# Min/Max open_time
		# --------
		minmax_df = pd.read_sql_query(
			f"SELECT MIN(open_time) as min_time, MAX(open_time) as max_time FROM {table_name}",
			conn
		)
		min_time = minmax_df["min_time"].iloc[0] or "—"
		max_time = minmax_df["max_time"].iloc[0] or "—"

		print(f"{table_name:<25} {row_count:<12} {str(min_time):<20} {str(max_time):<20}")

print(f"{'='*80}\n")


📊 DATABASE TABLES SUMMARY
Table Name                Rows         Min Date             Max Date            
--------------------------------------------------------------------------------
bchusdt_1m                3135156      2019-11-28 10:00:00  2025-11-15 06:19:00 
bchusdt_1m_features       4241076      2019-11-28 10:00:00  2025-11-15 06:19:00 
bchusdt_1m_predictions    3135398      2019-11-28 10:00:00  2025-11-15 06:19:00 



# Table info

In [3]:
# =============================================================================
# Database control: drop table and display tail
# =============================================================================
# Purpose:
#  - Drop table by name (no confirmation)
#  - Display last 5 rows from table
# =============================================================================

import sqlite3
import pandas as pd
import utils

# =============================================================================
# drop_table(table_name: str) -> None
# =============================================================================
# Purpose:
#  - Drop specified table from database
#  - Load db_path from config
#  - Direct drop without confirmation
# Parameters:
#  - table_name: name of table to drop
# =============================================================================
def drop_table(table_name: str) -> None:
	# -------------------------------------------------------------------------
	# Load config
	# -------------------------------------------------------------------------
	config  = utils._load_config()
	db_path = config["database"]["db_path"]

	# -------------------------------------------------------------------------
	# Drop table
	# -------------------------------------------------------------------------
	with sqlite3.connect(db_path) as conn:
		conn.execute(f"DROP TABLE IF EXISTS {table_name}")
		conn.commit()

	print(f"✅ Dropped table '{table_name}'")


# =============================================================================
# tail(table_name: str, n: int = 5) -> None
# =============================================================================
# Purpose:
#  - Query last n rows from table (SELECT * ORDER BY rowid DESC LIMIT n)
#  - Display as DataFrame
#  - Show row count and table info
# Parameters:
#  - table_name: name of table to query
#  - n: number of rows to display (default: 5)
# =============================================================================
def tail(table_name: str, n: int = 5) -> None:
	# -------------------------------------------------------------------------
	# Load config
	# -------------------------------------------------------------------------
	config  = utils._load_config()
	db_path = config["database"]["db_path"]

	# -------------------------------------------------------------------------
	# Query last n rows
	# -------------------------------------------------------------------------
	with sqlite3.connect(db_path) as conn:
		# Get total row count
		count_df = pd.read_sql_query(
			f"SELECT COUNT(*) as cnt FROM {table_name}",
			conn
		)
		total_rows = count_df["cnt"].iloc[0]

		# Get last n rows
		df = pd.read_sql_query(
			f"SELECT * FROM {table_name} ORDER BY rowid DESC LIMIT {n}",
			conn
		)

	# -------------------------------------------------------------------------
	# Display
	# -------------------------------------------------------------------------
	print(f"\n{'='*80}")
	print(f"📋 TABLE: {table_name}")
	print(f"{'='*80}")
	print(f"Total rows: {total_rows}")
	print(f"\n🔍 Last {n} rows:")
	print(f"{'-'*80}")
	print(df.to_string(index=False))
	print(f"{'='*80}\n")

In [4]:
tail("bchusdt_1m") 


📋 TABLE: bchusdt_1m
Total rows: 3135157

🔍 Last 5 rows:
--------------------------------------------------------------------------------
open_time_ms           open_time  open  high   low  close  volume
        None 2025-11-15 06:20:00 497.0 497.0 496.9  496.9   1.731
        None 2025-11-15 06:19:00 497.3 497.3 497.3  497.3   0.000
        None 2025-11-15 06:18:00 497.1 497.4 497.1  497.3   9.449
        None 2025-11-15 06:17:00 497.1 497.1 497.0  497.1  11.624
        None 2025-11-15 06:16:00 496.4 497.2 496.4  497.1   8.427



# Datatable validation

In [5]:
# =============================================================================
# Database control: validate table open_time integrity
# =============================================================================
# Purpose:
#  - Check if open_time column is unique
#  - Check if open_time is sorted ascending
#  - Check if each consecutive open_time is 1 minute apart
#  - Report any violations
# =============================================================================

import sqlite3
import pandas as pd
from datetime import datetime, timedelta
import utils

# =============================================================================
# validate_open_time(table_name: str) -> None
# =============================================================================
# Purpose:
#  - Load open_time from table sorted ascending
#  - Check uniqueness of open_time
#  - Check if each row is exactly 1 minute after previous
#  - Print validation results and any violations
# Parameters:
#  - table_name: name of table to validate
# =============================================================================
def validate_open_time(table_name: str) -> None:
	# -------------------------------------------------------------------------
	# Load config
	# -------------------------------------------------------------------------
	config  = utils._load_config()
	db_path = config["database"]["db_path"]

	# -------------------------------------------------------------------------
	# Query table
	# -------------------------------------------------------------------------
	with sqlite3.connect(db_path) as conn:
		df = pd.read_sql_query(
			f"SELECT open_time FROM {table_name} ORDER BY open_time ASC",
			conn
		)

	if df.empty:
		print(f"⚠️  Table '{table_name}' is empty")
		return

	open_times = df["open_time"].tolist()

	print(f"\n{'='*70}")
	print(f"🔍 VALIDATING TABLE: {table_name}")
	print(f"{'='*70}")

	# -------------------------------------------------------------------------
	# Check 1: Uniqueness
	# -------------------------------------------------------------------------
	unique_count = len(set(open_times))
	total_count  = len(open_times)

	print(f"\n✓ Total rows: {total_count}")
	print(f"✓ Unique open_time values: {unique_count}")

	if unique_count != total_count:
		print(f"❌ VIOLATION: open_time is NOT unique!")
		print(f"   Duplicates found: {total_count - unique_count}")
		return

	print(f"✅ open_time is unique")

	# -------------------------------------------------------------------------
	# Check 2: 1-minute intervals
	# -------------------------------------------------------------------------
	violations = []

	for i in range(len(open_times) - 1):
		current_str  = open_times[i]
		next_str     = open_times[i + 1]

		# Parse timestamps
		current = datetime.strptime(current_str, "%Y-%m-%d %H:%M:%S")
		next_ts = datetime.strptime(next_str, "%Y-%m-%d %H:%M:%S")

		# Calculate difference
		diff = next_ts - current

		# Check if exactly 1 minute (60 seconds)
		if diff != timedelta(minutes=1):
			violations.append({
				"index": i,
				"current": current_str,
				"next": next_str,
				"diff_seconds": int(diff.total_seconds())
			})

	if violations:
		print(f"❌ VIOLATION: {len(violations)} gaps are NOT 1 minute!")
		print(f"\n   Issues found:")
		for v in violations[:10]:  # Show first 10
			print(f"   Row {v['index']}: {v['current']} -> {v['next']} (diff: {v['diff_seconds']}s)")
		if len(violations) > 10:
			print(f"   ... and {len(violations) - 10} more")
	else:
		print(f"✅ All consecutive open_time values are exactly 1 minute apart")

	print(f"\n{'='*70}\n")

In [6]:
# -------------------------------------------------------------------------# Validate table
validate_open_time("bchusdt_1m")
# validate_open_time("bchusdt_1m_features")
# validate_open_time("bchusdt_1m_predictions")


🔍 VALIDATING TABLE: bchusdt_1m

✓ Total rows: 3115080
✓ Unique open_time values: 3115080
✅ open_time is unique
❌ VIOLATION: 17 gaps are NOT 1 minute!

   Issues found:
   Row 104639: 2020-02-09 01:59:00 -> 2020-02-09 03:00:00 (diff: 3660s)
   Row 119555: 2020-02-19 11:35:00 -> 2020-02-19 17:30:00 (diff: 21300s)
   Row 139227: 2020-03-04 09:21:00 -> 2020-03-04 11:30:00 (diff: 7740s)
   Row 213537: 2020-04-25 01:59:00 -> 2020-04-25 04:30:00 (diff: 9060s)
   Row 305547: 2020-06-28 01:59:00 -> 2020-06-28 05:30:00 (diff: 12660s)
   Row 528777: 2020-11-30 05:59:00 -> 2020-11-30 07:00:00 (diff: 3660s)
   Row 559425: 2020-12-21 13:47:00 -> 2020-12-21 18:00:00 (diff: 15180s)
   Row 564225: 2020-12-25 01:59:00 -> 2020-12-25 03:00:00 (diff: 3660s)
   Row 633386: 2021-02-11 03:40:00 -> 2021-02-11 05:00:00 (diff: 4800s)
   Row 666326: 2021-03-06 01:59:00 -> 2021-03-06 03:30:00 (diff: 5460s)
   ... and 7 more




# Drop tables

In [5]:
# =============================================================================
# Database control: drop table by name
# =============================================================================
# Purpose:
#  - Drop a table from database by name
#  - Load db_path from config
#  - No confirmation needed
# =============================================================================

import sqlite3
import utils

# =============================================================================
# drop_table(table_name: str) -> None
# =============================================================================
# Purpose:
#  - Drop specified table from database
#  - Load db_path from config
#  - Direct drop without confirmation
# Parameters:
#  - table_name: name of table to drop
# =============================================================================
def drop_table(table_name: str) -> None:
	# -------------------------------------------------------------------------
	# Load config
	# -------------------------------------------------------------------------
	config  = utils._load_config()
	db_path = config["database"]["db_path"]

	# -------------------------------------------------------------------------
	# Drop table
	# -------------------------------------------------------------------------
	with sqlite3.connect(db_path) as conn:
		conn.execute(f"DROP TABLE IF EXISTS {table_name}")
		conn.commit()

	print(f"✅ Dropped table '{table_name}'")

In [8]:
# Drop table
#drop_table("bchusdt_1m_predictions")
drop_table("bchusdt_1m_features")

✅ Dropped table 'bchusdt_1m_features'
